# Intelligent Agents: Reflex-Based Agents for GridHunt (Multi-agent Environment)

Student Name: [Add your name]

I have used the following AI tools: [list tools]

I understand that my submission needs to be my own work: [your initials]

## Learning Outcomes

* Apply core AI concepts by implementing the agent function for a simple and model-based reflex agents that respond to environmental percepts.
* Practice how the environment and the agent function interact.
* Analyze agent performance through controlled experiments across different environment configurations.

## Instructions

Total Points: 10

Complete this notebook. Use the provided notebook cells and insert additional code and markdown cells as needed. Submit the completely rendered notebook.
### AI Use

Here are some guidelines that will make it easier for you:

* __Don't:__ Rely on AI auto completion. You will waste a lot of time trying to figure out how the suggested code relates to what we do in class. Turn off AI code completion (e.g., Copilot) in your IDE.
* __Don't:__ Do not submit code/text that you do not understand or have not checked to make sure that it is complete and correct.
* __Do:__ Use AI for debugging and letting it explain code and concepts from class.

## Introduction

![GridHunt Image](gridhunt.png)

Gridhunt is a small educational game that helps you practice implementing simple agent functions. It is a modified Python reimplementation of Gridhunt2. 

The objective of gridhunt is to implement an intelligent agent, the hunter, who can catch a monster faster than all other hunters. Gridhunt uses a $n \times n$ arena consisting of a grid of tiles. Gridhunt is a turn-based game, and several hunter agents and the monster agent move around in the arena. A hunter catches the monster if she/he moves on the same square the monster currently occupies. The hunters compete fow who catches the monster first. If the monster survives the maximum number of steps for the game, then the monster wins.

## PEAS Description of Gridhunt

__Performance Measure:__ The performance is measured as the number of steps the hunter uses to catch the monster. Catching the monster means to be on the same square.

__Environment:__ An arena with $n \times n$ squares. At the beginning the monster and the hunters are randomly placed in the arena environment. The hunters and the monster
    can move around, but cannot leave the arena. Since there are three agents involved, this is a competitive multi-agent environment.

__Actuators:__ The agent can move to an adjacent square using actions "north", "east", "west", and "south", or teleport which will move the agent 
    to a random square in the arena.

__Sensors:__ The agent can always see its location in the arena and the location of the monster.  

## The Arena Environment

The environment manages the location of the agents (hunter and monster). It initially places them in a random location and then provides them in every step with percepts (the location of the hunter and the monster) and asks them for their action. 

__A Note on positions:__
The arena is implemented as an array with row and column indices representing the position.
Positions are stored in a numpy array where `pos[0]` is the row index and `pos[1]` is the column index in the arena. North means up in this array, that is the row index gets smaller when going north. Remember indices in Python start with 0 and go to `n-1`.

In [1]:
import numpy as np
from IPython.display import clear_output
from time import sleep

def arena_environment(hunter_1_agent_function, hunter_2_agent_function, monster_agent_function, n = 30, max_steps = 100, visualize = False, animation = False):
    """
    Simulate an arena where two hunter agent tries to catch a monster agent.

    Parameters:
    hunter_1_agent_function and hunter_2_agent_function (function): A function that takes the current positions of the hunter and monster
                                      and returns the next move for the hunter ('north', 'east', 'west', 'south', 'teleport').
    monster_agent_function (function): A function that takes the current positions of the hunter and monster
                                       and returns the next move for the monster (or 'stay' to remain in place).
    n (int): The size of the arena (n x n grid).
    max_steps (int): The maximum number of steps to simulate.
    visualize (bool): Whether to visualize the arena.
    animation (bool): Whether to animate the visualization with a delay.

    Returns:
    int: The number of steps taken for the hunter to catch the monster or np.nan (not a number) if not caught.
    """
    
    # Initialize positions
    monster_pos = np.random.randint(0, n-1, size=2)
    hunter_1_pos = np.random.randint(0, n-1, size=2)
    while np.array_equal(hunter_1_pos, monster_pos):
        hunter_1_pos = np.random.randint(0, n-1, size=2)
    hunter_2_pos = np.random.randint(0, n-1, size=2)
    while np.array_equal(hunter_2_pos, monster_pos):
        hunter_2_pos = np.random.randint(0, n-1, size=2)    
    
    def move(action, position):
        """calculate new position for the agent."""
        if action == 'north':
            position[0] -= 1
        elif action == 'south':
            position[0] += 1
        elif action == 'west':
            position[1] -= 1
        elif action == 'east':
            position[1] += 1
        else:
            raise ValueError("Invalid action.")
        
        # Ensure position stays within bounds
        position = np.clip(position, 0, n-1)
        return position

    def print_arena():

        if animation:
            sleep(1)
            clear_output(wait=True)
        print(f"Step {step}:")
        print(f"Hunter 1 is at '{hunter_1_pos}'; Hunter 2 is at '{hunter_2_pos}'; Monster is at '{monster_pos}'\n")
        arena = np.full((n, n), '.', dtype=str)
        arena[monster_pos[0], monster_pos[1]] = 'M'
        arena[hunter_1_pos[0], hunter_1_pos[1]] = '1'
        arena[hunter_2_pos[0], hunter_2_pos[1]] = '2'
        print("\n".join("".join(row) for row in arena))

    for step in range(max_steps):
        if visualize:
            print_arena()
        
        # Check if hunter has caught the monster
        if np.array_equal(hunter_1_pos, monster_pos):
            if visualize:
                print(f"Hunter 1 caught the monster in {step} steps!")
            return ("1", step)
        
        if np.array_equal(hunter_2_pos, monster_pos):
            if visualize:
                print(f"Hunter 2 caught the monster in {step} steps!")
            return ("2", step)

        # Get next move from monster agent function
        monster_action = monster_agent_function(monster_pos, hunter_1_pos, hunter_2_pos)
        if monster_action != 'stay':
            monster_pos = move(monster_action, monster_pos)

        # Get next move from hunter agent function
        hunter_1_action = hunter_1_agent_function(hunter_1_pos, monster_pos)
        if hunter_1_action == 'teleport':
            hunter_1_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_1_pos = move(hunter_1_action, hunter_1_pos)

        hunter_2_action = hunter_2_agent_function(hunter_2_pos, monster_pos)
        if hunter_2_action == 'teleport':
            hunter_2_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_2_pos = move(hunter_2_action, hunter_2_pos)
    
        # print the agents' actions
        if visualize:
            print(f"Hunter 1 chose action '{hunter_1_action}'")
            print(f"Hunter 2 chose action '{hunter_2_action}'")
            print(f"Monster chose action '{monster_action}'")
            print("\n" + "="*n + "\n")

    # the hunter failed to catch the monster within max_steps
    if visualize:
        print(f"Monster won by surviving {max_steps} steps.")
    
    return ("M", np.nan)

I implement the monster here as a simple agent that moves around randomly, but mostly stays in its place. You can use AI to get a detailed explanation of how the following code works.

In [2]:
monster_actions = ["north", "east", "west", "south", "stay"]

def monster_agent_function_simple(monster_pos, hunter_1_pos, hunter_2_pos):
    return np.random.choice(monster_actions, p=[0.125, 0.125, 0.125, 0.125, 0.5])

# The Hunter Agent

Your job is to implement a simple-reflex hunter agent that can catch the monster. Remember, your agent is implemented as an agent function that get percepts and needs to return a valid action. The actions are: 

In [3]:
actions = ["north", "east", "west", "south", "teleport"]

## A Simple Example Implementation

Here is a very simple hunter that just runs around randomly and hopes that it bumps into the monster. 

In [4]:
def simple_randomized_hunter_agent_function(hunter_location, monster_location):
    return np.random.choice(actions)

Ask the agent for an action.

In [5]:
simple_randomized_hunter_agent_function([0,0], [5,5])

np.str_('south')

## Experimenting With the Agent

We can place the monster and the hunter into the environment and run a simulation by calling the environment function.

In [6]:
hunter_1 = simple_randomized_hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

arena_environment(hunter_1, 
                  hunter_2, 
                  monster_agent_function_simple, 
                  n=5, max_steps=5, visualize=True, animation=False)

Step 0:
Hunter 1 is at '[1 2]'; Hunter 2 is at '[3 3]'; Monster is at '[0 2]'

..M..
..1..
.....
...2.
.....
Hunter 1 chose action 'east'
Hunter 2 chose action 'north'
Monster chose action 'east'

=====

Step 1:
Hunter 1 is at '[1 3]'; Hunter 2 is at '[2 3]'; Monster is at '[0 3]'

...M.
...1.
...2.
.....
.....
Hunter 1 chose action 'north'
Hunter 2 chose action 'south'
Monster chose action 'north'

=====

Step 2:
Hunter 1 is at '[0 3]'; Hunter 2 is at '[3 3]'; Monster is at '[0 3]'

...1.
.....
.....
...2.
.....
Hunter 1 caught the monster in 2 steps!


('1', 2)

Well, this was one run, maybe it was just good or bad luck this time!

Let's run an experiment with 100 runs in a $20 \times 20$ arena.  

In [7]:
hunter_1 = simple_randomized_hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

steps = [arena_environment(hunter_1, 
                           hunter_2, 
                           monster_agent_function_simple, 
                           n=20, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)

[('M', nan), ('2', 32), ('2', 22), ('M', nan), ('1', 31), ('2', 3), ('M', nan), ('M', nan), ('M', nan), ('2', 12), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 39), ('M', nan), ('M', nan), ('M', nan), ('2', 82), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 6), ('2', 46), ('1', 89), ('M', nan), ('1', 76), ('1', 3), ('M', nan), ('1', 79), ('M', nan), ('2', 34), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 71), ('M', nan), ('M', nan), ('M', nan), ('2', 19), ('M', nan), ('M', nan), ('1', 66), ('2', 19), ('M', nan), ('M', nan), ('2', 27), ('1', 51), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 90), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 17), ('M', nan), ('M', nan), ('M', nan), ('1', 17), ('1', 62), ('1', 27), ('1', 47), ('1', 42

We just got a list with who one and the number of steps to catch the monster for 100 simulation runs.
Let's analysis the results by counting who won how often.

In [8]:
steps = np.array(steps)

print (f"Hunter 1 won: {np.sum(steps[:,0] == "1")}")
print (f"Hunter 2 won: {np.sum(steps[:,0] == "2")}")
print (f"Monster 1 won: {np.sum(steps[:,0] == "M")}")       


Hunter 1 won: 18
Hunter 2 won: 15
Monster 1 won: 67


The random agent in not a very good hunter. You need to implement a better agent function.

# Your Agent Implementation [4 points]

Write a new hunter agent function that chases the monster. Copy the simple randomized hunter function from above and modify it using rules based on the monster's and the hunter's location. Let your hunter compete with the random agent. You can also come up with several different agents and let them compete against each other.

In [9]:
# here goes your agent implementation
def hunter_agent_function(hunter_location, monster_location):
    if monster_location is not None:
        if hunter_location[0] < monster_location[0]:
            return 'south'
        if hunter_location[0] > monster_location[0]:
            return 'north'
        if hunter_location[1] < monster_location[1]:
            return 'east'
        if hunter_location[1] > monster_location[1]:
            return 'west'
        else:
            return 'teleport'

# Your Experiments [2 points]

Copy the simulation code from above and run experiments with your agent. 
Experiment with larger arenas of at least size $30 \times 30$.

In [10]:
hunter_1 = hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

arena_environment(hunter_1, 
                  hunter_2, 
                  monster_agent_function_simple, 
                  n=5, max_steps=5, visualize=True, animation=False)

Step 0:
Hunter 1 is at '[1 1]'; Hunter 2 is at '[3 0]'; Monster is at '[1 3]'

.....
.1.M.
.....
2....
.....
Hunter 1 chose action 'east'
Hunter 2 chose action 'west'
Monster chose action 'stay'

=====

Step 1:
Hunter 1 is at '[1 2]'; Hunter 2 is at '[3 0]'; Monster is at '[1 3]'

.....
..1M.
.....
2....
.....
Hunter 1 chose action 'east'
Hunter 2 chose action 'north'
Monster chose action 'stay'

=====

Step 2:
Hunter 1 is at '[1 3]'; Hunter 2 is at '[2 0]'; Monster is at '[1 3]'

.....
...1.
2....
.....
.....
Hunter 1 caught the monster in 2 steps!


('1', 2)

In [11]:
# Experiment code (20x20 grid, 100 runs, max 100 steps)
hunter_1 = hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

steps = [arena_environment(hunter_1, 
                           hunter_2,    
                           monster_agent_function_simple, 
                           n=20, max_steps=40, visualize=False) for _ in range(100)] 
print(steps)

[('1', 6), ('1', 12), ('1', 12), ('1', 25), ('1', 14), ('1', 9), ('1', 14), ('1', 23), ('1', 16), ('1', 27), ('1', 21), ('1', 11), ('1', 15), ('1', 7), ('1', 33), ('M', nan), ('1', 26), ('1', 27), ('1', 19), ('1', 15), ('1', 19), ('1', 5), ('1', 6), ('1', 19), ('1', 13), ('1', 28), ('1', 15), ('1', 14), ('1', 6), ('1', 35), ('1', 5), ('1', 16), ('1', 8), ('1', 25), ('1', 13), ('1', 16), ('1', 18), ('1', 20), ('1', 20), ('1', 15), ('1', 22), ('M', nan), ('1', 33), ('1', 8), ('1', 15), ('1', 6), ('2', 6), ('1', 6), ('1', 4), ('1', 8), ('1', 9), ('1', 22), ('1', 33), ('1', 6), ('M', nan), ('1', 7), ('1', 10), ('M', nan), ('1', 11), ('1', 23), ('1', 20), ('1', 28), ('1', 8), ('1', 37), ('1', 12), ('1', 17), ('1', 26), ('1', 4), ('1', 13), ('1', 25), ('1', 21), ('1', 21), ('1', 17), ('1', 17), ('1', 23), ('1', 26), ('1', 21), ('1', 13), ('2', 7), ('1', 13), ('1', 4), ('1', 17), ('1', 28), ('1', 4), ('1', 17), ('1', 11), ('1', 11), ('1', 19), ('1', 3), ('1', 3), ('1', 21), ('1', 10), ('1', 2

In [12]:
# Summary of results
steps = np.array(steps)

print (f"Hunter 1 won: {np.sum(steps[:,0] == "1")}")
print (f"Hunter 2 won: {np.sum(steps[:,0] == "2")}")
print (f"Monster 1 won: {np.sum(steps[:,0] == "M")}")       

Hunter 1 won: 94
Hunter 2 won: 2
Monster 1 won: 4


In [13]:
# Experiment code (50x50 grid, 100 runs, max 100 steps)
hunter_1 = hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

steps = [arena_environment(hunter_1, 
                           hunter_2, 
                           monster_agent_function_simple, 
                           n=50, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)

# Summary of results
steps = np.array(steps)

print (f"Hunter 1 won: {np.sum(steps[:,0] == "1")}")
print (f"Hunter 2 won: {np.sum(steps[:,0] == "2")}")
print (f"Monster 1 won: {np.sum(steps[:,0] == "M")}")       

[('1', 46), ('1', 54), ('1', 23), ('1', 45), ('1', 39), ('1', 59), ('1', 32), ('1', 76), ('1', 51), ('1', 42), ('1', 47), ('1', 57), ('1', 8), ('1', 12), ('1', 50), ('2', 52), ('1', 19), ('1', 22), ('1', 33), ('1', 13), ('1', 50), ('1', 25), ('1', 62), ('1', 16), ('1', 12), ('1', 39), ('1', 6), ('1', 54), ('M', nan), ('1', 39), ('1', 13), ('1', 21), ('1', 55), ('1', 70), ('1', 23), ('1', 26), ('1', 24), ('1', 31), ('1', 60), ('1', 76), ('1', 55), ('1', 7), ('1', 54), ('1', 65), ('1', 58), ('1', 12), ('1', 31), ('1', 44), ('1', 41), ('1', 18), ('1', 46), ('M', nan), ('1', 57), ('1', 28), ('1', 31), ('1', 73), ('1', 49), ('1', 66), ('1', 36), ('1', 77), ('1', 45), ('1', 18), ('1', 61), ('1', 80), ('1', 44), ('1', 31), ('1', 7), ('1', 40), ('1', 24), ('1', 29), ('1', 33), ('1', 89), ('1', 66), ('1', 40), ('1', 99), ('M', nan), ('1', 38), ('1', 26), ('1', 14), ('1', 51), ('M', nan), ('1', 60), ('1', 50), ('1', 56), ('1', 58), ('1', 29), ('1', 20), ('1', 59), ('1', 12), ('1', 6), ('1', 74),

## Your Conclusion [4 points] 

Discuss the following:

* What is your hunter's final strategy to choose actions. Why does it work well?
* Do you use teleportation. Why and in what situation? Why not?
* How does the arena size affects your hunter agent's performance?

### Discussion
1. What is your hunter's final strategy to choose actions. Why does it work well?
The final strategy of Hunter 1 is to move directly toward the monster based on its current location. When the hunter receives the monster's position, it compares its own coordinates with the monster's coordinates and chooses the movement direction that reduces the distance between them.
Specifically:
- If the hunter is above the monster, it moves south.
- If the hunter is below the monster, it moves north.
- If the hunter is on the left side of the monster, it moves east.
- If the hunter is on the right side of the monster, it moves west.
When the hunter reaches the same position as the monster, teleportation is included as a fallback action. However, it is rarely used because the environment already checks whether the monster has been caught.
This strategy works well because Hunter 1 always moves toward the monster instead of moving randomly. By choosing actions that reduce the distance to the monster at each step, Hunter 1 can catch the monster more consistently. The experiment results show that the goal-directed strategy performs much better than random movement.

2. Do you use teleportation. Why and in what situation? Why not?
Yes, my hunter agent includes teleportation as an additional action, but it is not the main strategy. The agent mainly uses directional movements (north, south, east, and west) to reduce the distance between the hunter and the monster.  
Teleportation moves the hunter to a random position in the arena, so using it when the hunter is far away from the monster may increase the distance and waste previous progress. Therefore, teleportation is not preferred during normal chasing. It can only be considered as a fallback action in special situations, but direct movement is more reliable because it provides a predictable path toward the monster.

3. How does the arena size affect your hunter agent's performance?
The arena size affects the number of steps required for the hunter to catch the monster. In a smaller arena, the hunter and monster are usually closer together, so Hunter 1 can reach the monster faster.
When the arena becomes larger, the initial distance between the hunter and monster can increase. As a result, Hunter 1 needs more steps to move toward the monster.
However, the performance difference between Hunter 1 and Hunter 2 becomes more noticeable in larger arenas. The random hunter has a higher chance of moving away from the monster or wasting steps, while Hunter 1 continues following the shortest path toward the target. Therefore, increasing the arena size mainly increases the time needed to catch the monster rather than reducing the effectiveness of the goal-based strategy.

The monster is also an agent. Describe how the monster's agent function could be changed so it gets better at avoiding the hunter.

### Discussion
The monster agent could be improved by using the positions of both hunters to make better decisions. Instead of moving randomly, the monster can detect which hunter is closer and move away from that hunter to increase the distance. It can also avoid staying still and use unpredictable movements when the hunters are far away. This strategy would make the monster harder to catch and increase its survival time.

# More Work (optional)

* Improve the monster agent so it tries to run away from the hunters. Don;t make the monster too fast or the hunters will never be able to catch it.
* How could the two hunter agents work together? You need to think about information sharing. This can be implemented by the two agent functions sharing a single Python class or the environment could be expanded by an action "shout" that lets the other agent perceive a message.